E-18: selection by the held-out squared-loss Riesz criterion (`thm:selection_oracle`, `cor:selection_inference`)

Design document: `doc/2026-10-07_experiment_registration.md` (parent repository), E-18 and §1.2–§1.9; it is not a binding registration (U28), and the choices it leaves open are recorded in `docs/spec/02_data-and-experiments.md` §2.1. One notebook for every stage; the runner passes the stage:

- `python3 notebooks/experiments/run_registered.py E-18 stage0 20_E18_heldout_riesz_selection.ipynb`
- `... E-18 stage0.5 ...`, then `... E-18 stage1 ...` (each after the previous stage is committed).

The designs, the 65 candidates, the replication function and the aggregation are in `grrexp/e18.py`; the pilot and Stage 1 run the same aggregation. A solver that raises stops the notebook and the run is recorded as failed (§1.1).

In [1]:
import io
import json
import os
import time

import pandas as pd

from grrexp import env, outputs

gr = env.use_submodule_genriesz()
from grrexp import baselines, e18, parallel  # noqa: E402
from grrexp.seeds import CONFIRMATORY_ENTROPY, PILOT_ENTROPY, Seeds  # noqa: E402

STAGE = json.loads(os.environ[outputs.RUN_ENV])["stage"]
ARMS = [f"{c}-{e}" for c in e18.CONSTRUCTIONS for e in ("candidates", "SEL", "CTRL")]
rec = outputs.RunRecorder(18, STAGE, cells=None if STAGE == "stage0" else e18.CELLS,
                          arms=None if STAGE == "stage0" else ARMS)
STAGE

'stage0'

# Stage 0: design checks (moments, bounds, selection constants, radius guarantee, 18B correct specification)

In [2]:
if STAGE == "stage0":
    s0, stage0_warnings, _ = baselines.run_recording_warnings(e18.stage0)
    rec.write_stage0("e18_design", s0)
    print(json.dumps({d: s0["designs"][d]["checks"] for d in s0["designs"]}, indent=1))
    print(json.dumps({k: v["ok"] for k, v in s0["radius"].items()}), s0["ukl_18b"]["checks"])
    if not s0["ok"]:
        raise RuntimeError("E-18 design check failed")
    rec.finalize(R=None, n=None, warnings=int(sum(stage0_warnings.values())), failures=0)

{
 "18A": {
  "eta_bound_holds": true,
  "e_range": true,
  "max_abs_alpha0": true,
  "E_alpha0_sq": true,
  "Cm2": true
 },
 "18B": {
  "eta_bound_holds": true,
  "e_range": true,
  "max_abs_alpha0": true,
  "E_alpha0_sq": true,
  "Cm2": true
 }
}
{"SQ": true, "UKL1": true, "BKL1": true, "BP0": true} {'l1': True, 'represents_u0': True}


# The workflow after the replications (shared by Stage 0.5 and Stage 1)

`e18.aggregate`: the per-cell summary, H18-Thm (18A, Holm 0.01), H18-Inf (18B, Holm 0.01) and the H18-Emp intervals. Bootstrap call order (family 5, stream 4, one generator): for construction (a) then (b), the statistics (b) the 90% point of $\sqrt{n_\mathcal V}(\varrho_{\hat a}-\varrho^*)$ on 18A, (c) the median of $\varrho_{\hat a}/\varrho^*$ on 18A, (e) the median of $\varrho_{\hat a}$ on 18B; within a statistic the sample sizes in increasing order; a cell absent from the aggregation draws nothing.

# Stage 0.5: timing of the complete workflow, status counts, 1-versus-12-worker identity (no estimates kept)

In [3]:
if STAGE == "stage0.5":
    import matplotlib.pyplot as plt

    parallel.configure_worker()
    timing, serial, para, all_tasks = [], [], [], []
    for workers in (1, outputs.PILOT_WORKERS):
        for ci, cell in enumerate(e18.CELLS):
            t0 = time.perf_counter()
            tasks = e18.tasks_for(PILOT_ENTROPY, outputs.PILOT_REPS, cells=[ci])
            res = parallel.run_tasks(e18.replicate, tasks, workers)
            e18.aggregate(e18.raw_frame(tasks, res), Seeds(18, entropy=PILOT_ENTROPY))
            timing.append({"cell": cell, "workers": workers, "seconds": time.perf_counter() - t0})
            if workers == 1:
                serial += res
                all_tasks += tasks
            else:
                para += res
    digest = parallel.assert_bit_identical(serial, para, expected=len(all_tasks))
    raw = e18.raw_frame(all_tasks, serial)
    t0 = time.perf_counter()
    buf = io.StringIO()
    e18.aggregate(raw, Seeds(18, entropy=PILOT_ENTROPY)).to_csv(buf, index=False)
    buf.seek(0)
    tabs, fig, mac = e18.outputs_from_summary(pd.read_csv(buf))
    plt.close(fig)
    whole_seconds = time.perf_counter() - t0
    rows = []
    for _, r in raw.iterrows():
        for st, c in json.loads(r["statuses"]).items():
            rows.append({"cell": e18.CELLS[r["cell"]], "arm": f"{r['construction']}-candidates", "status": st, "count": c})
        for est in ("SEL", "CTRL"):
            if r["design"] == "18B":
                rows.append({"cell": e18.CELLS[r["cell"]], "arm": f"{r['construction']}-{est}", "status": r[f"{est}_status"], "count": 1})
    st = pd.DataFrame(rows)
    st["cell"] = st["cell"].map(json.dumps)
    status_df = st.groupby(["cell", "arm", "status"], sort=True)["count"].sum().reset_index()
    status_df["cell"] = status_df["cell"].map(json.loads)
    rec.write_pilot("timing.csv", pd.DataFrame(timing))
    rec.write_pilot("status_counts.csv", status_df)
    rec.record_parallel_identity(cells=e18.CELLS, reps=outputs.PILOT_REPS, n_jobs=outputs.PILOT_WORKERS,
                                 digest=digest)
    print(pd.DataFrame(timing).groupby("workers")["seconds"].sum(), whole_seconds)
    print(status_df.groupby(["arm", "status"])["count"].sum().to_string())
    rec.finalize(R={p: outputs.PILOT_REPS for p in e18.DESIGNS}, n={p: list(e18.N_VALUES) for p in e18.DESIGNS},
                 warnings=e18.total_warnings(raw), failures=e18.total_failures(raw),
                 extra={"whole_pilot_aggregation_table_and_figure_seconds": whole_seconds})
    del raw

# Stage 1

In [4]:
if STAGE == "stage1":
    parallel.configure_worker()
    tasks = e18.tasks_for(CONFIRMATORY_ENTROPY)
    t0 = time.perf_counter()
    results = parallel.run_tasks(e18.replicate, tasks, outputs.PILOT_WORKERS)
    print("seconds", time.perf_counter() - t0)
    raw = e18.raw_frame(tasks, results)
    rec.write_raw(raw)
    n_warn, n_fail = e18.total_warnings(raw), e18.total_failures(raw)
    raw.shape, n_warn, n_fail

In [5]:
if STAGE == "stage1":
    summ = e18.aggregate(raw, Seeds(18))
    rec.write_summary(summ)
    {k: json.loads(summ[k].iloc[0]) for k in ("thm_verdict", "thm_rejected", "inf_verdict", "inf_rejected", "emp_verdicts")}

# Table, figure and macros (from summary.csv only, §4)

In [6]:
if STAGE == "stage1":
    S = outputs.read_summary(rec)
    tabs, fig, mac = e18.outputs_from_summary(S)
    for name, tex in tabs.items():
        rec.write_table(name, tex)
    rec.save_figure(fig, "fig_E18_oracle")
    rec.write_macros(mac)
    print(mac)
    rec.finalize(R=dict(e18.R), n={p: list(e18.N_VALUES) for p in e18.DESIGNS}, warnings=int(n_warn),
                 failures=int(n_fail))